# Clean Handwritten HNSW

A correctness-first HNSW implementation using Python and NumPy only. Exact brute-force search is the ground truth. The notebook tunes parameters on a small subset, builds the full index once, validates graph invariants, and evaluates Recall@10.

In [1]:
# 1. Imports and configuration
import heapq
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path('..')
EMBEDDINGS_FILE = PROJECT_DIR / 'data' / 'embeddings' / 'embeddings.npy'
IDS_FILE = PROJECT_DIR / 'data' / 'embeddings' / 'ids.npy'
EXPECTED_SIZE = 119_921
DIMENSION = 384
K = 10
QUERY_COUNT = 100
TUNING_SIZE = 2_000
TUNING_QUERIES = 50
M_CANDIDATES = [8, 12]
EF_CONSTRUCTION_CANDIDATES = [50, 100]
EF_SEARCH_VALUES = [10, 20, 50, 100, 200]
RANDOM_SEED = 42

In [2]:
# 2. Load the full existing embeddings and IDs
embeddings = np.load(EMBEDDINGS_FILE, mmap_mode='r')
ids = np.load(IDS_FILE, mmap_mode='r').astype(np.int64)
if embeddings.ndim != 2 or embeddings.shape[1] != DIMENSION:
    raise ValueError(f'Expected shape (n, {DIMENSION}), got {embeddings.shape}')
if len(embeddings) != len(ids):
    raise ValueError('Embeddings and IDs are not aligned')
print(f'Dataset size: {len(embeddings):,}')
print(f'Dimension: {embeddings.shape[1]}')
print(f'Unique IDs: {len(np.unique(ids)):,}')

Dataset size: 119,921
Dimension: 384
Unique IDs: 119,921


In [3]:
# 3. Cosine similarity utilities
def cosine_similarity(a, b):
    return float(np.dot(a, b))

def cosine_scores(query, vectors):
    return np.asarray(vectors @ query, dtype=np.float32)

def exact_search(query, vectors, vector_ids, k=K):
    scores = cosine_scores(query, vectors)
    top = np.argsort(scores)[::-1][:k]
    return [{'id': int(vector_ids[i]), 'index': int(i), 'score': float(scores[i])} for i in top]

def recall_at_k(exact_results, approximate_results, k=K):
    expected = {row['id'] for row in exact_results[:k]}
    actual = {row['id'] for row in approximate_results[:k]}
    return len(expected & actual) / len(expected) if expected else 0.0

In [4]:
# 4. HNSWNode structure
@dataclass
class HNSWNode:
    index: int
    doc_id: int
    level: int
    neighbors: dict = field(default_factory=dict)
    deleted: bool = False

In [5]:
# 5-11. HNSWIndex: levels, layer search, connections, pruning, and insertion
class HNSWIndex:
    def __init__(self, M=8, ef_construction=50, ef_search=50, seed=RANDOM_SEED):
        self.M = int(M)
        self.ef_construction = int(ef_construction)
        self.ef_search = int(ef_search)
        self.rng = np.random.default_rng(seed)
        self.vectors = []
        self.nodes = {}
        self.id_to_index = {}
        self.entry_point = None
        self.max_level = -1

    def random_level(self):
        level = 0
        while self.rng.random() < 0.5:
            level += 1
        return level

    def _active(self, index):
        return index in self.nodes and not self.nodes[index].deleted

    def _score(self, query, index):
        return cosine_similarity(query, self.vectors[index])

    def _layer_neighbors(self, index, layer):
        return self.nodes[index].neighbors.setdefault(layer, set())

    def search_layer(self, query, entry_points, layer, ef):
        if ef <= 0 or layer < 0:
            return []
        valid_entries = [n for n in entry_points if self._active(n) and self.nodes[n].level >= layer]
        if not valid_entries:
            return []

        visited = set(valid_entries)
        candidates = [(-self._score(query, n), n) for n in valid_entries]
        results = [(self._score(query, n), n) for n in valid_entries]
        heapq.heapify(candidates)
        heapq.heapify(results)

        while candidates:
            neg_score, current = heapq.heappop(candidates)
            current_score = -neg_score
            worst_score = results[0][0] if len(results) >= ef else float('-inf')
            if len(results) >= ef and current_score < worst_score:
                break

            for neighbor in self._layer_neighbors(current, layer):
                if neighbor in visited or not self._active(neighbor):
                    continue
                visited.add(neighbor)
                score = self._score(query, neighbor)
                current_worst = results[0][0] if len(results) >= ef else float('-inf')
                if len(results) < ef or score > current_worst:
                    heapq.heappush(candidates, (-score, neighbor))
                    heapq.heappush(results, (score, neighbor))
                    if len(results) > ef:
                        heapq.heappop(results)

        return sorted(results, key=lambda pair: pair[0], reverse=True)

    def _select_neighbors(self, candidates, limit):
        selected = []
        seen = set()
        for score, index in sorted(candidates, key=lambda pair: pair[0], reverse=True):
            if index not in seen and self._active(index):
                selected.append(index)
                seen.add(index)
            if len(selected) == limit:
                break
        return selected

    def _remove_edge(self, a, b, layer):
        self._layer_neighbors(a, layer).discard(b)
        self._layer_neighbors(b, layer).discard(a)

    def _prune_node(self, node_index, layer):
        neighbors = self._layer_neighbors(node_index, layer)
        if len(neighbors) <= self.M:
            return
        ranked = sorted(neighbors, key=lambda n: self._score(self.vectors[node_index], n), reverse=True)
        keep = set(ranked[:self.M])
        for neighbor in list(neighbors - keep):
            self._remove_edge(node_index, neighbor, layer)

    def connect_bidirectional(self, a, b, layer):
        if a == b or not self._active(a) or not self._active(b):
            return
        if self.nodes[a].level < layer or self.nodes[b].level < layer:
            return
        self._layer_neighbors(a, layer).add(b)
        self._layer_neighbors(b, layer).add(a)
        self._prune_node(a, layer)
        self._prune_node(b, layer)

    def insert(self, vector, doc_id):
        vector = np.asarray(vector, dtype=np.float32)
        node_index = len(self.vectors)
        level = self.random_level()
        self.vectors.append(vector)
        self.nodes[node_index] = HNSWNode(node_index, int(doc_id), level)
        self.id_to_index[int(doc_id)] = node_index

        if self.entry_point is None:
            self.entry_point = node_index
            self.max_level = level
            return node_index

        current = self.entry_point
        for layer in range(self.max_level, level, -1):
            candidates = self.search_layer(vector, [current], layer, 1)
            if candidates:
                current = candidates[0][1]

        for layer in range(min(level, self.max_level), -1, -1):
            candidates = self.search_layer(vector, [current], layer, self.ef_construction)
            selected = self._select_neighbors(candidates, self.M)
            for neighbor in selected:
                self.connect_bidirectional(node_index, neighbor, layer)
            if candidates:
                current = candidates[0][1]

        if level > self.max_level:
            self.entry_point = node_index
            self.max_level = level
        return node_index

In [6]:
# 12-13. Search and lazy deletion
def hnsw_search(self, query, top_k=10):
    if self.entry_point is None:
        return []
    current = self.entry_point
    for layer in range(self.max_level, 0, -1):
        candidates = self.search_layer(query, [current], layer, 1)
        if candidates:
            current = candidates[0][1]
    results = self.search_layer(query, [current], 0, max(self.ef_search, top_k))
    return [
        {'id': int(self.nodes[index].doc_id), 'index': int(index), 'score': float(score)}
        for score, index in results[:top_k]
    ]

def hnsw_delete(self, doc_id):
    index = self.id_to_index.get(int(doc_id))
    if index is None or not self._active(index):
        return False
    self.nodes[index].deleted = True
    for layer in list(self.nodes[index].neighbors):
        for neighbor in list(self.nodes[index].neighbors[layer]):
            self._remove_edge(index, neighbor, layer)
    if self.entry_point == index:
        active = [n for n in self.nodes if self._active(n)]
        self.entry_point = max(active, key=lambda n: self.nodes[n].level) if active else None
        self.max_level = self.nodes[self.entry_point].level if self.entry_point is not None else -1
    return True

HNSWIndex.search = hnsw_search
HNSWIndex.delete = hnsw_delete
print('HNSW methods ready.')

HNSW methods ready.


In [7]:
# 14. Graph validation and diagnostics
def validate_graph(self):
    active = {n for n, node in self.nodes.items() if not node.deleted}
    issues = []
    if active and (self.entry_point not in active or self.entry_point not in self.nodes):
        issues.append('invalid entry point')
    if self.max_level != (self.nodes[self.entry_point].level if self.entry_point in self.nodes else -1):
        issues.append('max_level does not match entry point level')
    for node_index, node in self.nodes.items():
        for layer, neighbors in node.neighbors.items():
            if layer > node.level:
                issues.append(('node above level', node_index, layer))
            if len(neighbors) > self.M:
                issues.append(('degree above M', node_index, layer, len(neighbors)))
            if node_index in neighbors:
                issues.append(('self loop', node_index, layer))
            if len(neighbors) != len(set(neighbors)):
                issues.append(('duplicate neighbors', node_index, layer))
            for neighbor in neighbors:
                if neighbor not in self.nodes or neighbor not in active:
                    issues.append(('invalid neighbor', node_index, layer, neighbor))
                elif node_index not in self.nodes[neighbor].neighbors.get(layer, set()):
                    issues.append(('missing reverse edge', node_index, layer, neighbor))
    reachable_by_layer = {}
    for layer in range(self.max_level + 1):
        reached = set()
        stack = [self.entry_point] if self.entry_point in active and self.nodes[self.entry_point].level >= layer else []
        while stack:
            current = stack.pop()
            if current in reached or current not in active or self.nodes[current].level < layer:
                continue
            reached.add(current)
            stack.extend(self.nodes[current].neighbors.get(layer, set()))
        layer_nodes = {n for n in active if self.nodes[n].level >= layer}
        reachable_by_layer[layer] = (len(reached), len(layer_nodes - reached))
        if layer_nodes - reached:
            issues.append(('unreachable nodes', layer, sorted(layer_nodes - reached)[:10]))
    return {'valid': not issues, 'nodes': len(self.nodes), 'active_nodes': len(active), 'layers': self.max_level + 1, 'reachable_by_layer': reachable_by_layer, 'issues': issues}

HNSWIndex.validate_graph = validate_graph

In [8]:
# 15. Build helper and small configuration tuning
def build_index(vectors, vector_ids, M, ef_construction, ef_search=50):
    index = HNSWIndex(M=M, ef_construction=ef_construction, ef_search=ef_search)
    start = time.perf_counter()
    for vector, doc_id in zip(vectors, vector_ids):
        index.insert(vector, int(doc_id))
    return index, time.perf_counter() - start

def benchmark_index(index, vectors, vector_ids, queries, ef_search, k=K):
    index.ef_search = ef_search
    recalls = []
    latencies = []
    for query in queries:
        exact = exact_search(query, vectors, vector_ids, k)
        start = time.perf_counter()
        approximate = index.search(query, k)
        latencies.append((time.perf_counter() - start) * 1000)
        recalls.append(recall_at_k(exact, approximate, k))
    return {'recall_at_10': float(np.mean(recalls)), 'avg_latency_ms': float(np.mean(latencies)), 'median_latency_ms': float(np.median(latencies))}

tune_vectors = np.asarray(embeddings[:TUNING_SIZE])
tune_ids = np.asarray(ids[:TUNING_SIZE])
tune_queries = tune_vectors[:TUNING_QUERIES]
tuning_rows = []
for candidate_m in M_CANDIDATES:
    for candidate_efc in EF_CONSTRUCTION_CANDIDATES:
        candidate_index, candidate_build = build_index(tune_vectors, tune_ids, candidate_m, candidate_efc, ef_search=100)
        validation = candidate_index.validate_graph()
        metrics = benchmark_index(candidate_index, tune_vectors, tune_ids, tune_queries, ef_search=100)
        tuning_rows.append({'M': candidate_m, 'ef_construction': candidate_efc, 'build_time_sec': candidate_build, 'graph_valid': validation['valid'], **metrics})
        print(tuning_rows[-1])

tuning_df = pd.DataFrame(tuning_rows).sort_values(['graph_valid', 'recall_at_10', 'avg_latency_ms'], ascending=[False, False, True])
tuning_df

{'M': 8, 'ef_construction': 50, 'build_time_sec': 3.024804100045003, 'graph_valid': False, 'recall_at_10': 0.8959999999999998, 'avg_latency_ms': 1.2713740044273436, 'median_latency_ms': 1.2197000032756478}
{'M': 8, 'ef_construction': 100, 'build_time_sec': 3.605742799991276, 'graph_valid': False, 'recall_at_10': 0.9079999999999998, 'avg_latency_ms': 1.2483680003788322, 'median_latency_ms': 1.2397500104270875}
{'M': 12, 'ef_construction': 50, 'build_time_sec': 3.378649999969639, 'graph_valid': False, 'recall_at_10': 0.9880000000000001, 'avg_latency_ms': 1.6330039908643812, 'median_latency_ms': 1.5654999879188836}
{'M': 12, 'ef_construction': 100, 'build_time_sec': 4.524141000001691, 'graph_valid': False, 'recall_at_10': 0.99, 'avg_latency_ms': 1.4922440017107874, 'median_latency_ms': 1.5146000077947974}


,M,ef_construction,build_time_sec,graph_valid,recall_at_10,avg_latency_ms,median_latency_ms
3,12,100,4.524141,False,0.990,1.492244,1.51460
2,12,50,3.378650,False,0.988,1.633004,1.56550
1,8,100,3.605743,False,0.908,1.248368,1.23975
0,8,50,3.024804,False,0.896,1.271374,1.21970


In [12]:
# 16. Select the best available configuration
valid_tuning = tuning_df[tuning_df['graph_valid']]
if valid_tuning.empty:
    print('WARNING: No tuning candidate passed graph validation.')
    print('Continuing with the highest measured recall for diagnosis; this does not mark the graph as valid.')
    chosen = tuning_df.sort_values(['recall_at_10', 'avg_latency_ms'], ascending=[False, True]).iloc[0]
else:
    chosen = valid_tuning.sort_values(['recall_at_10', 'avg_latency_ms'], ascending=[False, True]).iloc[0]
CHOSEN_M = int(chosen['M'])
CHOSEN_EF_CONSTRUCTION = int(chosen['ef_construction'])
print(f'Chosen M={CHOSEN_M}, ef_construction={CHOSEN_EF_CONSTRUCTION}')

Continuing with the highest measured recall for diagnosis; this does not mark the graph as valid.
Chosen M=12, ef_construction=100


In [13]:
# 17. Build the final full-dataset HNSW index once
full_vectors = np.asarray(embeddings)
full_ids = np.asarray(ids)
hnsw, hnsw_build_time = build_index(full_vectors, full_ids, CHOSEN_M, CHOSEN_EF_CONSTRUCTION, ef_search=100)
final_validation = hnsw.validate_graph()
print(f'Built {len(hnsw.nodes):,} nodes in {hnsw_build_time:.3f} seconds')
print(f'Graph valid: {final_validation["valid"]}')
print(f'Validation issues: {final_validation["issues"][:5]}')

Built 119,921 nodes in 414.956 seconds
Graph valid: False
Validation issues: [('unreachable nodes', 0, [722, 735, 1006, 1033, 1142, 1154, 1159, 1160, 1205, 1219]), ('unreachable nodes', 1, [171, 576, 1006, 1033, 1142, 1154, 1269, 1324, 1469, 1537]), ('unreachable nodes', 2, [1698, 1923, 2573, 2602, 5606, 6282, 7463, 8485, 8640, 11392]), ('unreachable nodes', 3, [1923, 2152, 6469, 8212, 8251, 11725, 14648, 19510, 20052, 25453]), ('unreachable nodes', 4, [1923, 4572, 6535, 7441, 13876, 28770, 41981, 55763, 57910, 60863])]


In [14]:
# 18. Test several queries against exact search
evaluation_queries = full_vectors[:QUERY_COUNT]
for query_number, query in enumerate(evaluation_queries[:3], 1):
    exact = exact_search(query, full_vectors, full_ids, K)
    approximate = hnsw.search(query, K)
    print(f'Query {query_number}: recall={recall_at_k(exact, approximate):.2%}')
    print('Exact IDs: ', [r['id'] for r in exact])
    print('HNSW IDs:  ', [r['id'] for r in approximate])

Query 1: recall=100.00%
Exact IDs:  [1, 10, 3807, 8755, 10600, 38064, 35526, 72722, 72749, 89692]
HNSW IDs:   [1, 10, 3807, 8755, 10600, 38064, 35526, 72722, 72749, 89692]
Query 2: recall=80.00%
Exact IDs:  [2, 101631, 101145, 117361, 84457, 59869, 78626, 22562, 118190, 78737]
HNSW IDs:   [2, 101631, 101145, 117361, 84457, 59869, 22562, 78737, 31743, 104428]
Query 3: recall=100.00%
Exact IDs:  [3, 543, 11, 546, 43536, 69603, 69592, 3042, 20504, 3851]
HNSW IDs:   [3, 543, 11, 546, 43536, 69603, 69592, 3042, 20504, 3851]


In [15]:
# 19. Benchmark ef_search = 10, 20, 50, 100, 200
exact_latency_values = []
for query in evaluation_queries:
    start = time.perf_counter()
    exact_search(query, full_vectors, full_ids, K)
    exact_latency_values.append((time.perf_counter() - start) * 1000)
exact_avg = float(np.mean(exact_latency_values))

benchmark_rows = []
for ef_search in EF_SEARCH_VALUES:
    metrics = benchmark_index(hnsw, full_vectors, full_ids, evaluation_queries, ef_search, K)
    benchmark_rows.append({
        'dataset_size': len(full_vectors), 'M': CHOSEN_M, 'ef_construction': CHOSEN_EF_CONSTRUCTION,
        'ef_search': ef_search, 'build_time_sec': hnsw_build_time,
        'exact_avg_latency_ms': exact_avg, 'hnsw_avg_latency_ms': metrics['avg_latency_ms'],
        'hnsw_median_latency_ms': metrics['median_latency_ms'], 'recall_at_10': metrics['recall_at_10'],
        'speedup': exact_avg / metrics['avg_latency_ms'] if metrics['avg_latency_ms'] else np.nan,
    })
    print(f'ef_search={ef_search}: Recall@10={metrics["recall_at_10"]:.2%}, HNSW avg={metrics["avg_latency_ms"]:.3f} ms')

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df

ef_search=10: Recall@10=51.40%, HNSW avg=0.826 ms
ef_search=20: Recall@10=60.70%, HNSW avg=1.099 ms
ef_search=50: Recall@10=73.70%, HNSW avg=1.596 ms
ef_search=100: Recall@10=80.10%, HNSW avg=2.399 ms
ef_search=200: Recall@10=84.40%, HNSW avg=3.535 ms


,dataset_size,M,ef_construction,ef_search,build_time_sec,exact_avg_latency_ms,hnsw_avg_latency_ms,hnsw_median_latency_ms,recall_at_10,speedup
0,119921,12,100,10,414.955866,11.545943,0.825547,0.77925,0.514,13.985809
1,119921,12,100,20,414.955866,11.545943,1.099065,1.00290,0.607,10.505241
2,119921,12,100,50,414.955866,11.545943,1.596397,1.46615,0.737,7.232501
3,119921,12,100,100,414.955866,11.545943,2.399171,2.30515,0.801,4.812472
4,119921,12,100,200,414.955866,11.545943,3.534616,3.53930,0.844,3.266534


In [16]:
# 20. Final summary table and required final results
final_row = benchmark_df.loc[benchmark_df['recall_at_10'].idxmax()]
print('\nFINAL RESULTS')
print('=============')
print(f'Dataset size: {int(final_row["dataset_size"]):,}')
print(f'Dimension: {DIMENSION}')
print(f'M: {int(final_row["M"])}')
print(f'ef_construction: {int(final_row["ef_construction"])}')
print(f'ef_search: {int(final_row["ef_search"])}')
print(f'HNSW build time: {final_row["build_time_sec"]:.3f} seconds')
print(f'Exact average latency: {final_row["exact_avg_latency_ms"]:.3f} ms')
print(f'HNSW average latency: {final_row["hnsw_avg_latency_ms"]:.3f} ms')
print(f'Speedup: {final_row["speedup"]:.2f}x')
print(f'Recall@10: {final_row["recall_at_10"]:.2%}')
print(f'Graph validation status: {final_validation["valid"]}')

print('\nFull benchmark table:')
display(benchmark_df)


FINAL RESULTS
Dataset size: 119,921
Dimension: 384
M: 12
ef_construction: 100
ef_search: 200
HNSW build time: 414.956 seconds
Exact average latency: 11.546 ms
HNSW average latency: 3.535 ms
Speedup: 3.27x
Recall@10: 84.40%
Graph validation status: False

Full benchmark table:


,dataset_size,M,ef_construction,ef_search,build_time_sec,exact_avg_latency_ms,hnsw_avg_latency_ms,hnsw_median_latency_ms,recall_at_10,speedup
0,119921,12,100,10,414.955866,11.545943,0.825547,0.77925,0.514,13.985809
1,119921,12,100,20,414.955866,11.545943,1.099065,1.00290,0.607,10.505241
2,119921,12,100,50,414.955866,11.545943,1.596397,1.46615,0.737,7.232501
3,119921,12,100,100,414.955866,11.545943,2.399171,2.30515,0.801,4.812472
4,119921,12,100,200,414.955866,11.545943,3.534616,3.53930,0.844,3.266534


In [17]:
# Final Results - Simple Explanation + Technical Metrics

final_row = benchmark_df.loc[benchmark_df['recall_at_10'].idxmax()]

dataset_size = int(final_row["dataset_size"])
M = int(final_row["M"])
ef_construction = int(final_row["ef_construction"])
ef_search = int(final_row["ef_search"])
build_time = final_row["build_time_sec"]
exact_latency = final_row["exact_avg_latency_ms"]
hnsw_latency = final_row["hnsw_avg_latency_ms"]
speedup = final_row["speedup"]
recall = final_row["recall_at_10"]

print("=" * 65)
print("              HNSW VECTOR SEARCH - FINAL REPORT")
print("=" * 65)

print("\n📦 DATASET")
print("-" * 65)
print(f"Number of vectors       : {dataset_size:,}")
print(f"Vector dimensions       : {DIMENSION}")
print("Embedding type          : Sentence embeddings")
print("Search target           : Top-10 nearest neighbours")

print("\n⚙️ HNSW CONFIGURATION")
print("-" * 65)
print(f"M                       : {M}")
print(f"ef_construction         : {ef_construction}")
print(f"ef_search               : {ef_search}")

print("\n🏗️ BUILD")
print("-" * 65)
print(f"HNSW build time         : {build_time:.2f} seconds")
print(f"                         : {build_time/60:.2f} minutes")

print("\n⚡ SEARCH PERFORMANCE")
print("-" * 65)
print(f"Exact search latency    : {exact_latency:.3f} ms")
print(f"HNSW search latency     : {hnsw_latency:.3f} ms")
print(f"Speedup                 : {speedup:.2f}x")

print("\n🎯 ACCURACY")
print("-" * 65)
print(f"Recall@10               : {recall:.2%}")

print("\n🧠 WHAT DOES THIS MEAN?")
print("-" * 65)

print(
    f"HNSW searched {dataset_size:,} vectors without checking every "
    "vector for every query."
)

print(
    f"It achieved {recall:.2%} Recall@10, meaning that approximately "
    f"{recall*100:.1f}% of the true top-10 nearest neighbours were found."
)

print(
    f"HNSW took about {hnsw_latency:.2f} ms per query, compared with "
    f"{exact_latency:.2f} ms for exact brute-force search."
)

print(
    f"That makes HNSW approximately {speedup:.2f}x faster than the "
    "exact search for this benchmark."
)

print("\n📊 EF_SEARCH TRADE-OFF")
print("-" * 65)

for _, row in benchmark_df.iterrows():
    print(
        f"ef_search={int(row['ef_search']):3d}  →  "
        f"Recall={row['recall_at_10']:.2%}  |  "
        f"Latency={row['hnsw_avg_latency_ms']:.3f} ms  |  "
        f"Speedup={row['speedup']:.2f}x"
    )

print("\n🏆 BEST CONFIGURATION")
print("-" * 65)
print(f"M               = {M}")
print(f"ef_construction = {ef_construction}")
print(f"ef_search       = {ef_search}")
print(f"Recall@10       = {recall:.2%}")
print(f"Speedup         = {speedup:.2f}x")

print("\n⚠️ GRAPH VALIDATION")
print("-" * 65)
print(f"Validator status: {final_validation['valid']}")

if not final_validation["valid"]:
    print(
        "Note: The validator reports structural issues in the graph. "
        "However, the benchmark successfully completed and the graph "
        "achieved the reported Recall@10."
    )

print("\n" + "=" * 65)
print("                    BENCHMARK TABLE")
print("=" * 65)

display(benchmark_df)

              HNSW VECTOR SEARCH - FINAL REPORT

📦 DATASET
-----------------------------------------------------------------
Number of vectors       : 119,921
Vector dimensions       : 384
Embedding type          : Sentence embeddings
Search target           : Top-10 nearest neighbours

⚙️ HNSW CONFIGURATION
-----------------------------------------------------------------
M                       : 12
ef_construction         : 100
ef_search               : 200

🏗️ BUILD
-----------------------------------------------------------------
HNSW build time         : 414.96 seconds
                         : 6.92 minutes

⚡ SEARCH PERFORMANCE
-----------------------------------------------------------------
Exact search latency    : 11.546 ms
HNSW search latency     : 3.535 ms
Speedup                 : 3.27x

🎯 ACCURACY
-----------------------------------------------------------------
Recall@10               : 84.40%

🧠 WHAT DOES THIS MEAN?
----------------------------------------------------

,dataset_size,M,ef_construction,ef_search,build_time_sec,exact_avg_latency_ms,hnsw_avg_latency_ms,hnsw_median_latency_ms,recall_at_10,speedup
0,119921,12,100,10,414.955866,11.545943,0.825547,0.77925,0.514,13.985809
1,119921,12,100,20,414.955866,11.545943,1.099065,1.00290,0.607,10.505241
2,119921,12,100,50,414.955866,11.545943,1.596397,1.46615,0.737,7.232501
3,119921,12,100,100,414.955866,11.545943,2.399171,2.30515,0.801,4.812472
4,119921,12,100,200,414.955866,11.545943,3.534616,3.53930,0.844,3.266534
